# Oracle Hybrid Search Mode

This notebook demonstrates `retrieval_mode="hybrid_search"`.

What you should see:

1. A seed run calls Tavily and writes results into Oracle.
2. A local-only check proves Oracle can retrieve the persisted rows.
3. A hybrid run returns a mix of Oracle local rows and fresh Tavily rows.
4. The final cell deletes this notebook's demo rows so the notebook can be rerun cleanly.

## Setup

Run this cell first. It loads `.env`, connects to Oracle, prepares the demo table, and imports shared notebook helpers.

In [1]:
from pathlib import Path
import sys

helper_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "examples" / "oracle" / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate
        break

if helper_path is None:
    candidate = Path.cwd() / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate

if helper_path is None:
    raise RuntimeError("Could not find examples/oracle/demo_helpers.py. Start Jupyter from the repository root or examples/oracle.")

sys.path.insert(0, str(helper_path.parent))
from demo_helpers import *

print("Using helper:", helper_path)

Loaded environment from /Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/.env
Keeping proxy variables because TAVILY_USE_ENV_PROXY=1
Connected to Oracle. Table: TAVILY_DOCUMENTS
Table exists. Schema is ready.
Demo helpers are ready. Scores are ranking signals, not probabilities.
Using helper: /Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/examples/oracle/demo_helpers.py


## Choose the demo query

Edit only `HYBRID_QUERY` below when you want to try your own query. The cleanup cells use this same value, so reruns stay predictable.

In [2]:
HYBRID_QUERY = "Oracle Database vector search for Tavily hybrid retrieval"

## Start from a clean query-specific state

This removes only rows for `HYBRID_QUERY`, not the whole table.

In [3]:
delete_rows_for_query(HYBRID_QUERY)

Deleted 0 old demo rows for this query.


0

## 1. Seed Oracle with Tavily results

`max_local=0` forces this first call to use Tavily only. With `save_foreign=True`, those Tavily results are written into Oracle.

In [4]:
hybrid_client = make_client(
    "hybrid_search",
    persistence_depth="cache_plus_memory",
)

seed_results = hybrid_client.search(
    HYBRID_QUERY,
    max_results=3,
    max_local=0,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

show_results("Seed run: Tavily results persisted into Oracle", seed_results)
show_persisted_rows(HYBRID_QUERY, "Oracle rows after seed run")

### Seed run: Tavily results persisted into Oracle
`total=3` `origins={'foreign': 3}`

| rank | origin | score | preview |
| --- | --- | --- | --- |
| 1 | foreign | 0.7494 | A hybrid vector index inherits all the information retrieval capabilities of Oracle Text search indexes and leverages... |
| 2 | foreign | 0.7261 | Hybrid search is an advanced information retrieval technique that lets you search documents by keywords and vectors,... |
| 3 | foreign | 0.7053 | # Hybrid RAG with Tavily: Combining Static Knowledge and Dynamic Web Data. Hybrid RAG with Tavily blends the... |

### Oracle rows after seed run

| memory_scope | retrieval_mode | cache_hit | query_count | source_title | preview |
| --- | --- | --- | --- | --- | --- |
| cache_plus_memory | hybrid_search | 0 | 1 | Hybrid RAG with Tavily: Combining... | # Hybrid RAG with Tavily: Combining Static Knowledge and Dynamic Web Data. Hybrid RAG... |
| cache_plus_memory | hybrid_search | 0 | 1 | Understand Hybrid Vector Indexes | A hybrid vector index inherits all the information retrieval capabilities of Oracle... |
| cache_plus_memory | hybrid_search | 0 | 1 | Perform Hybrid Search - Oracle Help... | Hybrid search is an advanced information retrieval technique that lets you search... |

## 2. Prove Oracle can retrieve the saved rows

`max_foreign=0` prevents Tavily from being called. If results appear here, they came from Oracle.

In [5]:
local_results = hybrid_client.search(
    HYBRID_QUERY,
    max_results=3,
    max_local=3,
    max_foreign=0,
    save_foreign=False,
)

show_results("Oracle local-only check", local_results)

### Oracle local-only check
`total=3` `origins={'local': 3}`

| rank | origin | score | preview |
| --- | --- | --- | --- |
| 1 | local | 0.4414 | A hybrid vector index inherits all the information retrieval capabilities of Oracle Text search indexes and leverages... |
| 2 | local | 0.3283 | Hybrid search is an advanced information retrieval technique that lets you search documents by keywords and vectors,... |
| 3 | local | 0.2770 | # Hybrid RAG with Tavily: Combining Static Knowledge and Dynamic Web Data. Hybrid RAG with Tavily blends the... |

## 3. Run true hybrid search

This call allows both Oracle local rows and Tavily fresh rows, then ranks the combined set.

In [6]:
hybrid_results = hybrid_client.search(
    HYBRID_QUERY,
    max_results=5,
    max_local=3,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)

show_results("Hybrid search: Oracle + Tavily", hybrid_results)
show_persisted_rows(HYBRID_QUERY, "Oracle rows after hybrid run")

### Hybrid search: Oracle + Tavily
`total=5` `origins={'foreign': 3, 'local': 2}`

| rank | origin | score | preview |
| --- | --- | --- | --- |
| 1 | foreign | 0.7494 | A hybrid vector index inherits all the information retrieval capabilities of Oracle Text search indexes and leverages... |
| 2 | foreign | 0.7261 | Hybrid search is an advanced information retrieval technique that lets you search documents by keywords and vectors,... |
| 3 | foreign | 0.7053 | # Hybrid RAG with Tavily: Combining Static Knowledge and Dynamic Web Data. Hybrid RAG with Tavily blends the... |
| 4 | local | 0.4414 | A hybrid vector index inherits all the information retrieval capabilities of Oracle Text search indexes and leverages... |
| 5 | local | 0.3283 | Hybrid search is an advanced information retrieval technique that lets you search documents by keywords and vectors,... |

### Oracle rows after hybrid run

| memory_scope | retrieval_mode | cache_hit | query_count | source_title | preview |
| --- | --- | --- | --- | --- | --- |
| cache_plus_memory | hybrid_search | 0 | 1 | Hybrid RAG with Tavily: Combining... | # Hybrid RAG with Tavily: Combining Static Knowledge and Dynamic Web Data. Hybrid RAG... |
| cache_plus_memory | hybrid_search | 0 | 1 | Understand Hybrid Vector Indexes | A hybrid vector index inherits all the information retrieval capabilities of Oracle... |
| cache_plus_memory | hybrid_search | 0 | 1 | Perform Hybrid Search - Oracle Help... | Hybrid search is an advanced information retrieval technique that lets you search... |
| cache_plus_memory | hybrid_search | 0 | 1 | Hybrid RAG with Tavily: Combining... | # Hybrid RAG with Tavily: Combining Static Knowledge and Dynamic Web Data. Hybrid RAG... |
| cache_plus_memory | hybrid_search | 0 | 1 | Understand Hybrid Vector Indexes | A hybrid vector index inherits all the information retrieval capabilities of Oracle... |
| cache_plus_memory | hybrid_search | 0 | 1 | Perform Hybrid Search - Oracle Help... | Hybrid search is an advanced information retrieval technique that lets you search... |

## Cleanup

Run this at the end. It deletes this notebook's demo rows so the next full run starts with a Tavily seed again.

In [7]:
delete_rows_for_query(HYBRID_QUERY)
print("Cleaned hybrid-search demo rows. Re-run from the top to see the first call use Tavily again.")

Deleted 6 old demo rows for this query.
Cleaned hybrid-search demo rows. Re-run from the top to see the first call use Tavily again.
